# AEGIS RL Analysis

Portfolio-ready analysis of AEGIS trajectory logs. This notebook loads `trajectories/*.json`, validates each run with the `RunResult` Pydantic schema, computes reward features, and visualizes reliability, efficiency, and reward quality over time.

## Setup

Run from the repository root so imports resolve against local source.

In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cua_loop.rl import reward_from_attempt
from cua_loop.types import RunResult

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titleweight"] = "bold"

## Load and Validate Trajectories

Only files that validate as `RunResult` are included. Other JSON files, such as policy snapshots, are skipped.

In [ ]:
TRAJ_DIR = ROOT / "trajectories"
json_files = sorted(TRAJ_DIR.glob("*.json"))
runs: list[tuple[Path, RunResult]] = []
skipped: list[tuple[str, str]] = []

for path in json_files:
    try:
        runs.append((path, RunResult.model_validate_json(path.read_text())))
    except Exception as exc:
        skipped.append((path.name, type(exc).__name__))

print(f"Found {len(json_files)} JSON files in {TRAJ_DIR}")
print(f"Loaded {len(runs)} valid RunResult files; skipped {len(skipped)} files")
if skipped:
    display(pd.DataFrame(skipped, columns=["file", "reason"]))

## Flatten Runs Into Attempt-Level Rows

Reward is recomputed from `cua_loop.rl.reward_from_attempt` so the notebook stays aligned with the training code. Strategy labels are inferred from step messages when available; otherwise they are marked `unknown`.

In [ ]:
def infer_strategy(attempt) -> str:
    text_parts = []
    if attempt.trajectory.final_message:
        text_parts.append(attempt.trajectory.final_message)
    for step in attempt.trajectory.steps:
        if step.model_message:
            text_parts.append(step.model_message)
        if step.verification_reason:
            text_parts.append(step.verification_reason)
    text = "\n".join(text_parts)
    match = re.search(r"AEGIS RL strategy:\s*([a-zA-Z0-9_\-]+)", text)
    return match.group(1) if match else "unknown"

records = []
for run_idx, (path, run) in enumerate(runs):
    ts = pd.to_datetime(path.stat().st_mtime, unit="s")
    for attempt in run.attempts:
        steps = len(attempt.trajectory.steps)
        failed_steps = sum(step.verification_passed is False for step in attempt.trajectory.steps)
        blocked_steps = sum(step.blocked for step in attempt.trajectory.steps)
        records.append({
            "file": path.name,
            "timestamp": ts,
            "run_index": run_idx,
            "task": run.task,
            "url": run.url,
            "run_success": run.success,
            "attempt_index": attempt.attempt_index,
            "attempt_success": attempt.verifier.success,
            "schema_valid": attempt.verifier.schema_valid,
            "rows_extracted": attempt.verifier.rows_extracted,
            "steps": steps,
            "failed_verification_steps": failed_steps,
            "blocked_steps": blocked_steps,
            "duration_s": attempt.duration_s,
            "reward": reward_from_attempt(attempt),
            "strategy": infer_strategy(attempt),
            "reason": attempt.verifier.reason,
        })

df = pd.DataFrame(records)
if df.empty:
    print("No valid trajectory attempts yet. Run `uv run cua-loop ...` or `uv run aegis-rl ...` to generate logs.")
else:
    df = df.sort_values(["timestamp", "run_index", "attempt_index"]).reset_index(drop=True)
    display(df.head())
    display(df.describe(include="all"))

## Success Rate Over Time

In [ ]:
if not df.empty:
    run_df = df.groupby(["file", "timestamp", "run_index"], as_index=False).agg(run_success=("run_success", "max"))
    run_df = run_df.sort_values("timestamp").reset_index(drop=True)
    run_df["run_number"] = run_df.index + 1
    run_df["cumulative_success_rate"] = run_df["run_success"].expanding().mean()

    ax = sns.lineplot(data=run_df, x="run_number", y="cumulative_success_rate", marker="o")
    ax.set_title("AEGIS Success Rate Over Time")
    ax.set_xlabel("Run Number")
    ax.set_ylabel("Cumulative Success Rate")
    ax.set_ylim(0, 1.05)
    plt.show()
else:
    print("No data to plot.")

## Average Steps Per Attempt

In [ ]:
if not df.empty:
    steps_by_run = df.groupby("run_index", as_index=False).agg(avg_steps=("steps", "mean"), success_rate=("attempt_success", "mean"))
    fig, ax1 = plt.subplots(figsize=(12, 6))
    sns.barplot(data=steps_by_run, x="run_index", y="avg_steps", color="#4C78A8", ax=ax1)
    ax1.set_title("Average Steps Per Attempt by Run")
    ax1.set_xlabel("Run Index")
    ax1.set_ylabel("Average Steps")
    ax2 = ax1.twinx()
    sns.lineplot(data=steps_by_run, x="run_index", y="success_rate", color="#F58518", marker="o", ax=ax2)
    ax2.set_ylabel("Attempt Success Rate")
    ax2.set_ylim(0, 1.05)
    plt.show()
else:
    print("No data to plot.")

## Reward Distribution Per Strategy

In [ ]:
if not df.empty:
    plt.figure(figsize=(12, 6))
    ax = sns.violinplot(data=df, x="strategy", y="reward", inner="quartile", cut=0)
    sns.stripplot(data=df, x="strategy", y="reward", color="black", alpha=0.35, size=4, ax=ax)
    ax.set_title("Reward Distribution by Search Strategy")
    ax.set_xlabel("Strategy")
    ax.set_ylabel("Raw Reward")
    plt.xticks(rotation=20, ha="right")
    plt.show()
else:
    print("No data to plot.")

## Correlation Heatmap

Correlation between action count, reward, extracted rows, duration, blocked steps, and failed verification steps.

In [ ]:
if not df.empty:
    corr_cols = ["steps", "reward", "rows_extracted", "duration_s", "blocked_steps", "failed_verification_steps"]
    corr = df[corr_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, linewidths=0.5)
    ax.set_title("Trajectory Metrics Correlation Heatmap")
    plt.show()
else:
    print("No data to plot.")

## Strategy Leaderboard

In [ ]:
if not df.empty:
    leaderboard = (
        df.groupby("strategy", as_index=False)
        .agg(
            attempts=("strategy", "size"),
            success_rate=("attempt_success", "mean"),
            mean_reward=("reward", "mean"),
            median_steps=("steps", "median"),
            mean_rows=("rows_extracted", "mean"),
        )
        .sort_values(["mean_reward", "success_rate"], ascending=False)
    )
    display(leaderboard.style.format({"success_rate": "{:.1%}", "mean_reward": "{:.3f}", "median_steps": "{:.1f}", "mean_rows": "{:.1f}"}))
else:
    print("No strategy leaderboard yet.")